In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import numpy as np
import pandas as pd

import tensorflow as tf

from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array

In [3]:
BASE = "/content/drive/MyDrive/MultiModalProject/Flickr8k"

IMAGE_DIR = os.path.join(BASE, "Images")

TRAIN_CSV = os.path.join(BASE, "dataset_caption_train.csv")
VAL_CSV   = os.path.join(BASE, "dataset_caption_val.csv")
TEST_CSV  = os.path.join(BASE, "dataset_caption_test.csv")

In [4]:
train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(29112, 2)
(3238, 2)
(8095, 2)


In [5]:
model = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    pooling=None
)

model.trainable = False

print("Model Loaded")

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Model Loaded


In [6]:
IMG_SIZE = (224,224)

def extract_feature(img_path):

    img = load_img(img_path, target_size=IMG_SIZE)

    img = img_to_array(img)

    img = np.expand_dims(img,0)

    img = preprocess_input(img)

    feature = model.predict(img,verbose=0)[0]

    feature = feature.reshape(-1,1280)

    return feature

In [7]:
def build_features(df):

    cache = {}

    features = {}

    for i,row in df.iterrows():

        image_name = row["image"]

        if image_name not in cache:

            path = os.path.join(IMAGE_DIR,image_name)

            cache[image_name] = extract_feature(path)

        features[image_name] = cache[image_name]

        if (i+1)%500==0:
            print(i+1)

    return features

In [8]:
train_features = build_features(train_df)

print(len(train_features))

500
1000
1500
2000
2500
3000
3500
4000
4500
5000
5500
6000
6500
7000
7500
8000
8500
9000
9500
10000
10500
11000
11500
12000
12500
13000
13500
14000
14500
15000
15500
16000
16500
17000
17500
18000
18500
19000
19500
20000
20500
21000
21500
22000
22500
23000
23500
24000
24500
25000
25500
26000
26500
27000
27500
28000
28500
29000
5824


In [9]:
val_features = build_features(val_df)

print(len(val_features))

500
1000
1500
2000
2500
3000
648


In [10]:
test_features = build_features(test_df)

print(len(test_features))

500
1000
1500
2000
2500
3000
3500
4000
4500
5000
5500
6000
6500
7000
7500
8000
1619


In [11]:
np.save(
    os.path.join(BASE,"efficient_train.npy"),
    train_features,
    allow_pickle=True
)

np.save(
    os.path.join(BASE,"efficient_val.npy"),
    val_features,
    allow_pickle=True
)

np.save(
    os.path.join(BASE,"efficient_test.npy"),
    test_features,
    allow_pickle=True
)

In [12]:
train = np.load(
    os.path.join(BASE,"efficient_train.npy"),
    allow_pickle=True
).item()

print(len(train))

first_key = list(train.keys())[0]

print(first_key)

print(train[first_key].shape)

5824
1000268201_693b08cb0e.jpg
(49, 1280)


In [13]:
train_features = np.load(
    os.path.join(BASE, "efficient_train.npy"),
    allow_pickle=True
).item()

image_name = train_df.iloc[0]["image"]

feature = train_features[image_name]

print(image_name)
print(feature.shape)

1000268201_693b08cb0e.jpg
(49, 1280)
